### T evaluation


In [ ]:
import os
import sys
from pathlib import Path

# Must be set before importing torch (or any module that imports it).
# Physical GPU index, or "" for CPU-only.
os.environ["CUDA_VISIBLE_DEVICES"] = input("GPU index: ")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "settings.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "models" / "modified-classification"
ROOT = NOTEBOOK_DIR.parents[1]
for p in (ROOT, NOTEBOOK_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from _helpers import (
    grid_axes_from_df,
    load_live_settings,
    load_run_settings,
    resolve_run_dir,
    score_metric_grids,
    timestep_pairs_from_df,
)

# Resolve run from live settings (DIR_NAME / OUTPUTS_DIR), then load that run's snapshot.
_live = load_live_settings()
# Optional override: RUN_DIR = NOTEBOOK_DIR / "outputs" / "..." / "20260709_144025"
RUN_DIR = resolve_run_dir(_live.OUTPUTS_DIR)
load_run_settings(RUN_DIR)
print(f"Loaded settings from {RUN_DIR / 'settings.py'}")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from model_m import SurrogateModel
from model_t import (
    TimestepSelector,
    gate_metrics,
    load_timestep_selector,
    per_image_spearman,
    regret,
)
from _data import (
    df_to_metric_grids,
    load_split_df,
)
from embeddings import get_embeddings_by_sample
from settings import *

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}  (CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES', '')!r})")


In [ ]:
# Load dataset splits.
PLOT_SPLIT = "test"
splits = load_split_df(RUN_DIR)
full_df = pd.concat(splits.values(), ignore_index=True)
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]
print(f"Run: {RUN_DIR.name}  " + "  ".join(f"{name}={len(df)} cells ({df[SAMPLE_ID_COL].nunique()} samples)" for name, df in splits.items()))

# Grid axes and labeled pairs come from the data: under TARGET_T_DELTA the grid
# is usually sparse (e.g. only t_start > t_end), so unlabeled cells stay NaN and
# every comparison below is restricted to the labeled pairs.
T_START_VALUES, T_END_VALUES = grid_axes_from_df(full_df)
T_PAIRS = timestep_pairs_from_df(full_df)
N1, N2 = len(T_START_VALUES), len(T_END_VALUES)
CELLS = len(T_PAIRS)
I_OF = {float(v): i for i, v in enumerate(T_START_VALUES)}
J_OF = {float(v): j for j, v in enumerate(T_END_VALUES)}
T_START_DELTA = int(np.argmin(np.abs(T_START_VALUES - DEFAULT_T_START)))
T_END_DELTA = int(np.argmin(np.abs(T_END_VALUES - DEFAULT_T_END)))
BASELINE_IDX = T_START_DELTA * N2 + T_END_DELTA
print(f"Labeled timestep pairs: {CELLS} of {N1 * N2} axis cells ({CELLS / (N1 * N2):.0%})")
print(
    f"default cell ({T_START_DELTA}, {T_END_DELTA}) -> "
    f"t_start={T_START_VALUES[T_START_DELTA]:.3g}, t_end={T_END_VALUES[T_END_DELTA]:.3g}"
)


In [ ]:
# Report current run name.
print(f"Run: {RUN_DIR.name} in {RUN_DIR.parent.name}")


In [ ]:

true_df = full_df.sort_values([SAMPLE_ID_COL, T_START_COL, T_END_COL]).reset_index(drop=True)
true_df[T_TARGET_COL] = T_TARGET_PHI_DF(true_df, *M_TARGET_COLS)
true_argmax_df = true_df.loc[true_df.groupby(SAMPLE_ID_COL)[T_TARGET_COL].idxmax()].reset_index(drop=True)
print(f"Found metrics for {len(true_df)} grid cells for {true_df[SAMPLE_ID_COL].nunique()} samples")
print(f"Found (t_start, t_end) argmax for {T_TARGET_LABEL} ({T_TARGET_COL}) for {len(true_argmax_df)} samples")
true_argmax_df.head()


In [ ]:
# Build T on the axes present in the data so predicted grids line up with the
# true grids from df_to_metric_grids.
timestep_selector = load_timestep_selector(RUN_DIR / "regressor_weights.pt", device=DEVICE)
timestep_selector = TimestepSelector(
    timestep_selector.surrogate,
    t_start_values=T_START_VALUES,
    t_end_values=T_END_VALUES,
)
input_embeddings = get_embeddings_by_sample(full_df, timestep_selector.surrogate, DEVICE)

# Restrict predictions to the labeled pairs so pred_df and true_df cover exactly
# the same cells; scoring unlabeled cells would let argmax pick a cell that has
# no ground truth to compare against.
pred_rows = []
for sid, emb in input_embeddings.items():
    grid = timestep_selector.predict_grid_from_emb(
        emb["img"], emb["mask"], emb["src"], emb["tar"], t_pairs=T_PAIRS,
    )
    for ts, te in T_PAIRS:
        i, j = I_OF[float(ts)], J_OF[float(te)]
        pred_rows.append({
            SAMPLE_ID_COL: sid,
            T_START_COL: float(ts),
            T_END_COL: float(te),
            PSNR_COL: grid.psnr_grid[i, j],
            CLIP_COL: grid.clip_grid[i, j],
        })

pred_df = pd.DataFrame(pred_rows).sort_values([SAMPLE_ID_COL, T_START_COL, T_END_COL]).reset_index(drop=True)
pred_df[T_TARGET_COL] = T_TARGET_PHI_DF(pred_df, *M_TARGET_COLS)
print(f"Predicted metrics for {len(pred_df)} grid cells for {pred_df[SAMPLE_ID_COL].nunique()} samples")
pred_argmax_df = pred_df.loc[pred_df.groupby(SAMPLE_ID_COL)[T_TARGET_COL].idxmax()].reset_index(drop=True)
print(f"Predicted (t_start, t_end) argmax for {T_TARGET_LABEL} ({T_TARGET_COL}) for {len(pred_argmax_df)} samples")
pred_argmax_df.head()


In [ ]:
eval_df = splits[PLOT_SPLIT]
SAMPLE_IDS = sorted(eval_df[SAMPLE_ID_COL].unique())
N_IMG = len(SAMPLE_IDS)

true_psnr, _ = df_to_metric_grids(eval_df, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, PSNR_COL)
true_clip, _ = df_to_metric_grids(eval_df, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, CLIP_COL)
true_phi = score_metric_grids(true_psnr, true_clip, BASELINE_IDX)

pred_psnr = np.full_like(true_psnr, np.nan)
pred_clip = np.full_like(true_clip, np.nan)
pred_phi = np.full_like(true_phi, np.nan)
for k, sid in enumerate(SAMPLE_IDS):
    emb = input_embeddings[sid]
    grid = timestep_selector.predict_grid_from_emb(
        emb["img"], emb["mask"], emb["src"], emb["tar"], t_pairs=T_PAIRS,
    )
    pred_psnr[k], pred_clip[k], pred_phi[k] = grid.psnr_grid, grid.clip_grid, grid.phi_grid

print(
    f"{PLOT_SPLIT} grids: {true_phi.shape}  ({N_IMG} images, "
    f"{CELLS} labeled pairs / {N1 * N2} axis cells)"
)


In [ ]:
def fig_title(subtitle: str, split: str) -> str:
    return f"{subtitle} on {METRICS_CSV.stem} for {T_TARGET_LABEL}, $\\delta={{{TARGET_T_DELTA:.2f}}}$ ({split})"


def best_first(grid: np.ndarray) -> np.ndarray:
    """Flat cell indices of one grid, best first, unlabeled (NaN) cells last.

    argsort places NaN last ascending, so a plain argsort()[::-1] would rank
    unlabeled cells FIRST; map NaN to -inf so they always sort to the back.
    """
    return np.argsort(np.nan_to_num(grid.ravel(), nan=-np.inf))[::-1]


In [ ]:
"""
Plot row-normalised confusion matrices for both heads across all three splits. Each cell
shows the raw count as a number and is shaded by the fraction of true-class samples that
fell in that predicted bucket, making it easy to see whether errors cluster near the
diagonal (near-miss) or scatter widely (large mistakes).
"""
from sklearn.metrics import confusion_matrix
from matplotlib.colors import LinearSegmentedColormap

cmp_df = true_argmax_df[[SAMPLE_ID_COL, T_START_COL, T_END_COL]].merge(
    pred_argmax_df[[SAMPLE_ID_COL, T_START_COL, T_END_COL]],
    on=SAMPLE_ID_COL,
    suffixes=("_true", "_pred"),
)
for col in (T_START_COL, T_END_COL):
    cmp_df[f"{col}_true"] = cmp_df[f"{col}_true"].astype(float)
    cmp_df[f"{col}_pred"] = cmp_df[f"{col}_pred"].astype(float)

BUCKET_LABELS_START = [f"{v:.1f}" for v in GRID_T_START]
BUCKET_LABELS_END = [f"{v:.1f}" for v in GRID_T_END]
t_start_to_idx = {float(v): i for i, v in enumerate(GRID_T_START)}
t_end_to_idx = {float(v): i for i, v in enumerate(GRID_T_END)}

results = {}
for split_name, split_df in splits.items():
    split_cmp = cmp_df[cmp_df[SAMPLE_ID_COL].isin(split_df[SAMPLE_ID_COL].unique())]
    results[split_name] = {
        "true_start": split_cmp[f"{T_START_COL}_true"].map(t_start_to_idx).to_numpy(),
        "pred_start": split_cmp[f"{T_START_COL}_pred"].map(t_start_to_idx).to_numpy(),
        "true_end": split_cmp[f"{T_END_COL}_true"].map(t_end_to_idx).to_numpy(),
        "pred_end": split_cmp[f"{T_END_COL}_pred"].map(t_end_to_idx).to_numpy(),
    }

cmap = LinearSegmentedColormap.from_list("wb", ["white", "steelblue"])

def plot_confusion(ax, true, pred, title, bucket_labels):
    # Plot confusion matrix for t_start and t_end
    n_buckets = len(bucket_labels)
    cm = confusion_matrix(true, pred, labels=list(range(n_buckets)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    im = ax.imshow(cm_norm, cmap=cmap, vmin=0, vmax=1)
    for i in range(n_buckets):
        for j in range(n_buckets):
            ax.text(
                j, i, f"{cm[i, j]}",
                ha="center", va="center", fontsize=9,
                color="white" if cm_norm[i, j] > 0.5 else "black",
            )
    ax.set_xticks(range(n_buckets))
    ax.set_yticks(range(n_buckets))
    ax.set_xticklabels(bucket_labels)
    ax.set_yticklabels(bucket_labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    return im

def plot_split(split):
    # Create separate confusion matrices for t_start and t_end
    r = results[split]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
    plot_confusion(axes[0], r["true_start"], r["pred_start"], f"t_start", BUCKET_LABELS_START)
    plot_confusion(axes[1], r["true_end"], r["pred_end"], f"t_end", BUCKET_LABELS_END)
    for ax in axes:
        ax.tick_params(axis="y", labelleft=True)
        ax.set_ylabel("True")
    fig.suptitle(fig_title("Confusion matrices", split), fontsize=14)
    fig.subplots_adjust(wspace=0.12, top=0.85)
    plt.show()
    plt.close(fig)

for split in ["train", "test"]:
    plot_split(split)

In [ ]:
# Top-N accuracy, is the true-best cell in the model's top-N predictions by T_TARGET_COL?

def true_best_in_top_n_pct(true_df, pred_df, n_values):
    """Percent of samples whose oracle argmax is inside pred top-N (by T_TARGET_COL)."""
    true_best = true_df.loc[
        true_df.groupby(SAMPLE_ID_COL)[T_TARGET_COL].idxmax(),
        [SAMPLE_ID_COL, T_START_COL, T_END_COL],
    ]
    # Rank each sample's cells once, then every N is a comparison on that rank.
    ranked = pred_df.sort_values(T_TARGET_COL, ascending=False)
    pred_order = {
        sid: list(zip(g[T_START_COL].astype(float), g[T_END_COL].astype(float)))
        for sid, g in ranked.groupby(SAMPLE_ID_COL, sort=False)
    }
    ranks = []
    for row in true_best.itertuples():
        cells = pred_order[getattr(row, SAMPLE_ID_COL)]
        true_cell = (float(getattr(row, T_START_COL)), float(getattr(row, T_END_COL)))
        ranks.append(cells.index(true_cell) + 1 if true_cell in cells else len(cells) + 1)
    ranks = np.array(ranks)
    return np.array([100.0 * np.mean(ranks <= n) for n in n_values])


def plot_top_n_recall(split_name, split_df):
    split_ids = split_df[SAMPLE_ID_COL].unique()
    split_true_df = true_df[true_df[SAMPLE_ID_COL].isin(split_ids)]
    split_pred_df = pred_df[pred_df[SAMPLE_ID_COL].isin(split_ids)]

    top_k = np.arange(1, 11)
    n_cells = split_pred_df.groupby(SAMPLE_ID_COL).size().iloc[0]
    top_k_acc = true_best_in_top_n_pct(split_true_df, split_pred_df, top_k)
    x_labels = [f"{n}" for n in top_k] # \n({100.0 * n / n_cells:.1f}%)

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(top_k, top_k_acc, color="steelblue", edgecolor="white", width=0.7)
    ax.set_xlabel(f"Considered Top-N by {T_TARGET_LABEL}")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xticks(top_k)
    ax.set_xticklabels(x_labels)
    ax.set_ylim(0, min(100, top_k_acc.max() + 15))
    for bar, pct in zip(bars, top_k_acc):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{pct:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    fig.suptitle(fig_title("Top-N recall", split_name), fontsize=14)
    plt.tight_layout()
    plt.show()


for split_name, split_df in [("train", train_df), ("test", test_df)]:
    plot_top_n_recall(split_name, split_df)


In [ ]:
# Metric summaries on test, plus deltas vs default.
from IPython.display import display

METRICS = [PSNR_COL, CLIP_COL, T_TARGET_COL]
STATS = ["mean", "median", "std", "min", "max"]
test_ids = set(test_df[SAMPLE_ID_COL].unique())

def metric_rows(df):
    out = df[df[SAMPLE_ID_COL].isin(test_ids)][[SAMPLE_ID_COL, *METRICS]].copy()
    out[METRICS] = out[METRICS].apply(pd.to_numeric)
    return out.set_index(SAMPLE_ID_COL)

default = true_df[
    np.isclose(true_df[T_START_COL].astype(float), DEFAULT_T_START)
    & np.isclose(true_df[T_END_COL].astype(float), DEFAULT_T_END)
]
default, true_best, pred_best = map(metric_rows, [default, true_argmax_df, pred_argmax_df])

# Create deltas columns.
selections = {"Default": default, "True Best": true_best, "Pred Best": pred_best}
values = {name: df.agg(STATS) for name, df in selections.items()}
deltas = {
    "Δ True Best": (true_best - default).agg(STATS),
    "Δ Pred Best": (pred_best - default).agg(STATS),
    "Δ Pred True": (pred_best - true_best).agg(STATS),
}
columns = ["Default", "True Best", "Δ True Best", "Pred Best", "Δ Pred Best", "Δ Pred True"]

print(fig_title("Comparative stats", "test"))
for metric in METRICS:
    # Create a table with the specified columns.
    table = pd.DataFrame({
        "Default": values["Default"][metric],
        "True Best": values["True Best"][metric],
        "Δ True Best": deltas["Δ True Best"][metric],
        "Pred Best": values["Pred Best"][metric],
        "Δ Pred Best": deltas["Δ Pred Best"][metric],
        "Δ Pred True": deltas["Δ Pred True"][metric],
    })[columns]
    # Format the table with the specified column formatting.
    print("\n", metric)
    display(table.style.format({
        col: ("{:+.3f}" if col.startswith("Δ") else "{:.3f}") for col in columns
    }))

In [ ]:

"""
Per-image Spearman rho - does M_hat rank grid cells correctly within each image?

For each image, correlate the ranked order of true vs predicted values across
all labeled cells. Reported for PSNR, CLIP, and scalarized phi separately.

Interpretation:
  - median rho > 0.5   -> strong within-image ranking; T argmax is well-founded.
  - median rho 0.2-0.5 -> weak but nonzero signal; expect moderate regret.
  - median rho < 0.2   -> near-random ranking; T selection is unreliable.
  - frac rho < 0       -> fraction of images where M_hat ranks backwards; should be ~0.
  - rho(phi) matters most since T argmax operates on phi, not raw PSNR/CLIP alone.
  - High rho(PSNR) but low rho(CLIP) -> check scalarization weights in settings.py.
"""

rho_psnr = per_image_spearman(true_psnr, pred_psnr)
rho_clip = per_image_spearman(true_clip, pred_clip)
rho_phi = per_image_spearman(true_phi, pred_phi)
for name, r in [("PSNR", rho_psnr), ("CLIP", rho_clip), ("phi", rho_phi)]:
    rv = r[np.isfinite(r)]
    print(f"{name:5s}  median rho={np.nanmedian(r):.3f}  frac<0.2={np.mean(rv < 0.2):.2f}  frac<0={np.mean(rv < 0):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, r) in zip(axes, [("PSNR", rho_psnr), ("CLIP", rho_clip), ("phi", rho_phi)]):
    rv = r[np.isfinite(r)]
    if len(rv): ax.hist(rv, bins=min(30, max(3, len(rv))), edgecolor="white")
    ax.axvline(0, color="k", lw=0.5); ax.set_title(f"rho: {name}"); ax.set_xlabel("Spearman rho")
fig.suptitle(fig_title("Per-image Spearman", PLOT_SPLIT), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""
Selection regret - how much true phi do we lose by following argmax(pred_phi)?

regret = true_phi[oracle cell] - true_phi[cell chosen by T]
Lower is better. Zero means T picked the true-best cell.

Interpretation:
  - median regret ~ 0, frac~0 high  -> T matches oracle on most images.
  - median regret < noise floor    -> gains are within label noise; defaults may suffice.
  - median regret >> noise floor   -> T is leaving real quality on the table.
  - p90 regret much larger than median -> a few images have catastrophic mis-selection.
  - Compare regret median to NOISE_FLOOR_PHI to judge practical significance.
"""

reg = regret(true_phi, pred_phi)
print(f"regret  median={np.median(reg):.4f}  p90={np.percentile(reg, 90):.4f}  frac~0={np.mean(reg < 1e-3):.2f}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(reg, bins=min(30, max(5, N_IMG // 2)), edgecolor="white")
ax.set_xlabel("regret (phi units)"); ax.set_ylabel("count"); ax.set_title("selection regret")
fig.suptitle(fig_title("Selection regret", PLOT_SPLIT), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""
Top-1 hit rate and top-3 overlap - does argmax(pred) land on the true best cell?

top-1 hit rate: fraction of images where pred argmax == true argmax (exact match).
top-3 overlap:  fraction of the true top-3 cells that also appear in pred top-3.

Interpretation:
  - hit rate well above 1/CELLS        -> better than chance; promising.
  - hit rate > 0.5                     -> strong exact-match performance.
  - high top-3 overlap, low hit rate   -> M_hat finds the right region but not exact peak.
  - both near 0                        -> ranking failure; check Spearman and surface plots.
  - top-3 overlap is more forgiving; useful when near-optimal cells are interchangeable.
"""

def topk_metrics(true_phi_grid, pred_phi_grid, k=3):
    hit1, ov = [], []
    for i in range(true_phi_grid.shape[0]):
        t_order = best_first(true_phi_grid[i])
        p_order = best_first(pred_phi_grid[i])
        hit1.append(t_order[0] == p_order[0])
        ov.append(len(set(t_order[:k]) & set(p_order[:k])) / k)
    return np.mean(hit1), np.mean(ov)

h1, ov3 = topk_metrics(true_phi, pred_phi, k=3)
print(f"top-1 hit rate={h1:.3f}   top-3 overlap={ov3:.3f}")


### Top-N recall — is the true-best cell among T's top-N predicted outputs?

For each N, plot the percent of images whose oracle-best `(t_start, t_end)` cell appears in T's top-N cells ranked by predicted `phi`.

**Interpretation**
- Steep rise in the first few N → T ranks the true best near the top quickly.
- Flat near 0 for small N → ranking failure; true best is rarely highly ranked.
- At N=1, equals top-1 hit rate; at N=121, must reach 100%.

In [ ]:
def oracle_in_top_n_pct(true_phi_grid, pred_phi_grid, n_values):
    """Percent of images with oracle argmax inside pred top-N (by phi)."""
    ranks = []
    for i in range(true_phi_grid.shape[0]):
        p_order = best_first(pred_phi_grid[i])
        oracle = best_first(true_phi_grid[i])[0]
        ranks.append(int(np.flatnonzero(p_order == oracle)[0]) + 1)
    ranks = np.array(ranks)
    return np.array([100.0 * np.mean(ranks <= n) for n in n_values])


In [ ]:
n_values = np.arange(1, CELLS + 1)
top_n_pct = oracle_in_top_n_pct(true_phi, pred_phi, n_values)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_values, top_n_pct, marker="o", markersize=3, linewidth=1.5)
ax.set_xlabel("True Best in Top-N Predictions")
ax.set_ylabel("Percent of Images")
ax.set_xlim(1, CELLS)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3)
for n_mark in sorted({1, 3, 5, 10, 25, CELLS} & set(range(1, CELLS + 1))):
    pct = top_n_pct[n_mark - 1]
    ax.axvline(n_mark, color="k", linestyle=":", alpha=0.2)
    ax.annotate(f"N={n_mark}: {pct:.0f}%", xy=(n_mark, pct), xytext=(4, 4),
                textcoords="offset points", fontsize=8)
fig.suptitle(fig_title("Oracle in Top-N Predictions", PLOT_SPLIT), fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
print("  ".join(f"top-{n}={top_n_pct[n - 1]:.1f}%" for n in (1, 3, 10, 25) if n <= CELLS))

In [ ]:

"""
Argmax collapse - does T pick diverse cells or always the same (t_start, t_end)?

Compare the spatial spread of true vs predicted argmax cells across images.
Red X marks the paper-default cell (t_start=0.9, t_end=0.3).

Interpretation:
  - pred spread ~ true spread, many unique pred cells -> T adapts per image (good).
  - pred spread << true spread                       -> collapse; M_hat surface is too flat.
  - unique pred cells = 1-2 out of CELLS             -> T ignores image content entirely.
  - pred mass on default cell while true is spread   -> gate may be too conservative.
  - pred mass at grid edges while true is interior   -> M_hat may overfit boundary artifacts.
"""

def argmax_cells(grid):
    """Best labeled cell per image; nanargmax so NaN cells are never selected."""
    return np.array([np.unravel_index(np.nanargmax(grid[k]), grid[k].shape) for k in range(grid.shape[0])])

true_arg = argmax_cells(true_phi)
pred_arg = argmax_cells(pred_phi)

def spread(arg):
    return arg[:, 0].std() + arg[:, 1].std()

print(f"true argmax spread = {spread(true_arg):.3f}")
print(f"pred argmax spread = {spread(pred_arg):.3f}")
print(f"unique pred cells = {len(set(map(tuple, pred_arg)))} / {CELLS}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
for ax, arg, title in zip(axes, [true_arg, pred_arg], ["true argmax", "pred argmax"]):
    H = np.zeros((N1, N2))
    for i, j in arg: H[i, j] += 1
    ax.imshow(H.T, origin="lower", cmap="Blues"); ax.set_title(title)
    ax.scatter([T_START_DELTA], [T_END_DELTA], c="r", marker="x", label="default")
    ax.set_xlabel("t_start idx"); ax.set_ylabel("t_end idx"); ax.legend(fontsize=8)
fig.suptitle(fig_title("Argmax collapse", PLOT_SPLIT), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:
nf = NOISE_FLOOR_PHI
gate = gate_metrics(true_phi, pred_phi, T_START_DELTA, T_END_DELTA, nf)
print(
    f"gate  precision={gate['precision']:.3f}  recall={gate['recall']:.3f}  "
    f"({gate['n_flagged']} flagged of {N_IMG})  noise_floor={nf:.4f}"
)
precision = gate["precision"]
recall = gate["recall"]


In [ ]:

"""
Error stratification - where does T fail?

Relates regret to default-cell quality and whether the true optimum sits on the
grid boundary (t_start or t_end at 0.0 or 1.0).

Interpretation:
  - regret vs default_quality correlation near 0 -> errors are not driven by baseline quality.
  - higher regret when best_at_edge=True       -> M_hat struggles at grid boundaries; expected if
      training data is sparse near edges.
  - higher regret when best_at_edge=False      -> interior ranking failure; more concerning.
  - scatter cluster at high regret, low default -> images where defaults are poor and T also fails.
"""

df_err = pd.DataFrame({
    "sample_id": SAMPLE_IDS,
    "regret": reg,
    "default_quality": true_phi[:, T_START_DELTA, T_END_DELTA],
    "best_at_edge": [(i in (0, N1-1)) or (j in (0, N2-1)) for i, j in true_arg],
})
# corrcoef needs >= 2 images; a single-sample split would just warn and return NaN.
corr = float(np.corrcoef(df_err["default_quality"], df_err["regret"])[0, 1]) if N_IMG > 1 else float("nan")
print(f"regret vs default quality corr: {corr:.3f}" + ("" if N_IMG > 1 else "  (needs >1 image)"))
print("regret by edge:", df_err.groupby("best_at_edge")["regret"].median().to_dict())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(df_err["default_quality"], df_err["regret"], alpha=0.7)
axes[0].set_xlabel("default phi"); axes[0].set_ylabel("regret")
axes[1].boxplot([df_err.loc[~df_err.best_at_edge, "regret"], df_err.loc[df_err.best_at_edge, "regret"]], tick_labels=["interior", "edge"])
axes[1].set_ylabel("regret")
fig.suptitle(fig_title("Error stratification", PLOT_SPLIT), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""
T evaluation summary - key decision metrics in one place.

Use this cell to decide whether to deploy T with the current M_hat checkpoint:
  - Spearman phi median > 0.4 AND regret median < noise floor  -> deploy with gate.
  - Argmax collapse (spread(pred) << spread(true))             -> do not deploy; fix M_hat first.
  - Gate recall < 0.5 with many improvable images              -> lower noise floor or retrain M_hat.
"""

print(fig_title("T summary", PLOT_SPLIT))
print(f"  split: {PLOT_SPLIT}  images: {N_IMG}  grid: {N1}x{N2}")
print(f"  Spearman phi median: {np.nanmedian(rho_phi):.3f}")
print(f"  regret median:     {np.median(reg):.4f}")
print(f"  top-1 / top-3:     {h1:.3f} / {ov3:.3f}")
print(f"  spread true/pred:  {spread(true_arg):.3f} / {spread(pred_arg):.3f}")
print(f"  gate prec/recall:  {precision:.3f} / {recall:.3f}")
